# A large scale analysis of conversation data across myriad spoken corpora

In [1]:
import os
import re
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from datetime import datetime as dt

import warnings; warnings.filterwarnings('ignore')

In [2]:
DATA_LOC = '../data'

# path to processed and ready data
DATA_PATH = os.path.join(DATA_LOC, 'lme_ready')

# path to save data to on completion
VIS_PATH = os.path.join(DATA_LOC, 'web_vis')

# post processing & ready for data visualization steps
FINAL_DOC = os.path.join(DATA_LOC, 'resids.parquet')

In [3]:
def grab_all_files(PATH, file_type='.csv'):
    files = [
        [
            os.path.join(root, f) for f in files 
            if f.endswith(file_type) and (not f.startswith('._'))
        ] 
        for root, _, files in os.walk(PATH) 
    ]
    return sum(files, [])

### Load data

In [4]:
# df = pd.read_table(FINAL_DOC, sep='\t')
df = pd.read_parquet(FINAL_DOC)
df.shape

(25425942, 14)

In [5]:
df.isna().sum()

Hxy                0
nx                 0
ny                 0
turn_delta         0
self               0
x_speaker          0
dyad               0
corpus             0
conversation_id    0
x_turn_id          0
null               0
groups             0
sample_delta       0
resid              0
dtype: int64

In [6]:
df['conversation_id'].nunique()

2662

In [7]:
(df['turn_delta'] < 0).sum()

np.int64(60066)

In [8]:
df['corpus'].loc[(df['turn_delta'] < 0)].unique()

array(['se0-ag1', 'se0-ag2', 'se0-ag3', 'se0-ag4', 'se1-ag1', 'se1-ag3',
       'se2-ag1', 'se2-ag3', 'se2-ag4', 'se3-ag1', 'se3-ag2', 'se3-ag3'],
      dtype=object)

In [9]:
df.head()

,Hxy,nx,ny,turn_delta,self,x_speaker,dyad,corpus,conversation_id,x_turn_id,null,groups,sample_delta,resid
0,6.344819,20.0,6.0,1,False,GCSAusE-GCSAusE-01-S,GCSAusE-GCSAusE-01-S-GCSAusE-GCSAusE-01-H,GCSAusE,GCSAusE-GCSAusE-01,GCSAusE-GCSAusE-01-6,False,GCSAusE-GCSAusE-01-GCSAusE-GCSAusE-01-6,1,0.355656
1,6.000488,20.0,8.0,2,False,GCSAusE-GCSAusE-01-S,GCSAusE-GCSAusE-01-S-GCSAusE-GCSAusE-01-H,GCSAusE,GCSAusE-GCSAusE-01,GCSAusE-GCSAusE-01-6,False,GCSAusE-GCSAusE-01-GCSAusE-GCSAusE-01-6,2,0.035690
2,5.521488,20.0,13.0,4,False,GCSAusE-GCSAusE-01-S,GCSAusE-GCSAusE-01-S-GCSAusE-GCSAusE-01-H,GCSAusE,GCSAusE-GCSAusE-01,GCSAusE-GCSAusE-01-6,False,GCSAusE-GCSAusE-01-GCSAusE-GCSAusE-01-6,3,-0.382399
3,6.057978,20.0,6.0,6,False,GCSAusE-GCSAusE-01-S,GCSAusE-GCSAusE-01-S-GCSAusE-GCSAusE-01-H,GCSAusE,GCSAusE-GCSAusE-01,GCSAusE-GCSAusE-01-6,False,GCSAusE-GCSAusE-01-GCSAusE-GCSAusE-01-6,4,0.068815
4,6.198739,20.0,6.0,7,False,GCSAusE-GCSAusE-01-S,GCSAusE-GCSAusE-01-S-GCSAusE-GCSAusE-01-H,GCSAusE,GCSAusE-GCSAusE-01,GCSAusE-GCSAusE-01-6,False,GCSAusE-GCSAusE-01-GCSAusE-GCSAusE-01-6,5,0.209576


### Convert values to numeric tags

In [ ]:
convert_to_numeric_id = [
    #'x_speaker', 'y_speaker', 
    # 'dyad', 
    #'x_speaker_turn'
]

In [ ]:
for col in convert_to_numeric_id:
    vals = np.unique(df[col].values)
    conversion = {k: i+1 for i,k in enumerate(np.random.choice(vals, size=(len(vals),), replace=False))}
    
    if isinstance(col,list):
        for c  in col:
            df[c] = [conversion[v] for v in tqdm(df[c])]
    
    else:
        df[col] = [conversion[v] for v in tqdm(df[col])]


### Other data preparation processes

In [ ]:
df.head()

## LME Regression: Predicting CE

In [10]:
import statsmodels.formula.api as smf

In [ ]:
df_ = df.loc[~df['null']]
df_.shape

In [12]:
df_['null'].value_counts()

null
False    23233133
True      2192809
Name: count, dtype: int64

### Time-inclusive model

In [ ]:
##########################################
## Main model
##########################################
model = "Hxy ~ nx + ny + sample_delta*self"
##########################################

In [ ]:
start = dt.now()
# md = smf.mixedlm(model, data=df_, groups=df_['x_speaker_turn'])
md = smf.mixedlm(model, data=df_, groups=df_['x_turn_id'])
mdf = md.fit()
print('completed in:', dt.now()-start)

In [ ]:
# mdf.summary()

In [ ]:
reporting = pd.DataFrame()
reporting['coefs'] = mdf.params
reporting['stat'] = mdf.tvalues
reporting['p'] = mdf.pvalues
reporting['CI[.025, .975]'] = ['[{}]'.format(', '.join([np.format_float_scientific(x, precision=2) for x in ci.tolist()])) for ci in mdf.conf_int().values]

reporting['coefs'] = reporting['coefs'].apply(lambda x: np.format_float_scientific(x, precision=2))
reporting['stat'] = reporting['stat'].apply(lambda x: np.format_float_scientific(x, precision=2))
reporting['p'] = reporting['p'].apply(lambda x: np.format_float_scientific(x, precision=2))

reporting.head(100)

Saving results for reporting

In [ ]:
lme_results_name = os.path.join(DATA_LOC, 'lme-results-with-coraal-20250812.csv')

In [ ]:
reporting.to_csv(lme_results_name, encoding='utf-8')

In [ ]:
reporting['Var'] = reporting.index.values

with open(lme_results_name.replace('.csv', '2.txt'), 'w') as f:
    txt =  reporting[['Var', 'coefs', 'stat', 'p']].to_latex(index=False).replace('\\toprule', '\\hline').replace('\\midrule', '\\hline\\hline').replace('\\bottomrule', '\\hline')

    txt = txt.replace('\\\\', '\\\\\hline').replace('|', '$|$').replace('_delta', ' $\Delta$')
    f.write(txt)
    f.close()